In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
try:
    from IPython.display import display
except ImportError:
    display = print

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
sns.set_style('whitegrid')

In [ ]:
# Load the unified dataset
data_path = Path('data/collection/unified_dataset.jsonl')
df = pd.read_json(data_path, lines=True)

# Preview the dataframe
print(f"Dataset shape: {df.shape}")
display(df.head(3))

In [ ]:
# Schema and data types
print("Data types:")
print(df.dtypes)

# Basic info
print(f"\nTotal rows: {len(df)}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# Missingness analysis
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'non_null': len(df) - missing, 'null': missing, 'null_pct': missing_pct})
print("Missingness summary:")
display(missing_df)

# Visualize missingness
fig, ax = plt.subplots(figsize=(8, 4))
missing_df['null_pct'].plot(kind='bar', ax=ax, color='coral')
ax.set_title('Missing Values Percentage by Column')
ax.set_ylabel('Percentage')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Label distribution (only 10 labels present out of 30)
if 'label' in df.columns and df['label'].notna().any():
    label_counts = df['label'].value_counts()
    print(f"Label column: {df['label'].notna().sum()} non-null values out of {len(df)}")
    print(f"Unique labels: {df['label'].nunique()}")
    
    fig, ax = plt.subplots(figsize=(10, 5))
    label_counts.plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title('Label Distribution')
    ax.set_ylabel('Count')
    ax.set_xticklabels([f'Label {i}' for i in range(len(label_counts))], rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No label column or all null")

In [ ]:
# Text length distribution
if 'text' in df.columns and df['text'].notna().any():
    df['text_length'] = df['text'].str.split().str.len()
    
    print("Text length statistics (words):")
    print(df['text_length'].describe())
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Histogram
    axes[0].hist(df['text_length'], bins=15, color='teal', edgecolor='white')
    axes[0].set_title('Text Length Distribution')
    axes[0].set_xlabel('Word Count')
    axes[0].set_ylabel('Frequency')
    
    # Box plot
    axes[1].boxplot(df['text_length'], vert=True)
    axes[1].set_title('Text Length Box Plot')
    axes[1].set_ylabel('Word Count')
    
    plt.tight_layout()
    plt.show()
else:
    print("No text column or all null")

In [ ]:
# Source distribution
if 'source' in df.columns:
    source_counts = df['source'].value_counts()
    print("Source distribution:")
    print(source_counts)
    
    fig, ax = plt.subplots(figsize=(8, 4))
    source_counts.plot(kind='bar', ax=ax, color='darkgreen')
    ax.set_title('Source Distribution')
    ax.set_ylabel('Count')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
else:
    print("No source column")

## Data Quality Review

### Analyzer View

- Task interpretation: unknown
- Primary modality: text
- Target semantics: unknown
- Relevant checks: text_completeness_all_30_rows_have_valid_text, label_format_verification_labels_use_chain_of_thought_format, source_balance_three_sources_10_each, text_length_distribution_math_problem_typical_lengths
- Lower-value checks: class_balance_plots_not_useful_for_numeric_math_answers, outlier_detection_not_applicable_text_domain, imputation_for_missing_labels_not_recommended, audio_column_analysis_all_null
- Priority actions: drop_audio_column, preserve_unlabeled_rows_for_potential_label_generation, keep_image_urls_as_optional_reference

### Strategy Justification

This is a math reasoning dataset where the 'label' column contains chain-of-thought solutions, not simple class labels. The 20 rows with null labels are intentionally unlabeled math problems from project-euler and all-russian sources - NOT missing data to impute. Generic imputation strategies like 'median' are inappropriate because: (1) these are text solutions, not numeric values; (2) unlabeled rows are valid problems for inference or future label generation. The strategy preserves all 30 rows, drops the useless audio column, and flags unlabeled rows in metadata for downstream processing. This maximizes utility for math problem solving, curriculum learning, or active learning pipelines.

- Missing values: `no_imputation`
- Duplicates: `no_deduplication`
- Outliers: `not_applicable_text_domain`

### Findings

- Missing values before cleaning: 70
- Duplicate rows before cleaning: 0
- Numeric outliers before cleaning: 0
- Imbalance column: `label`
- Majority class share after cleaning: not applicable

### Before / After

- Missing values: 0 -> 0
- Duplicates: 0 -> 0
- Outliers: 0 -> 0


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

raw_df = pd.read_json('data/collection/unified_dataset.jsonl', lines=True)
clean_df = pd.read_json('data/quality/cleaned_dataset.jsonl', lines=True)
primary_modality = 'text'

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
missing_counts = raw_df.isna().sum().sort_values(ascending=False)
missing_counts = missing_counts[missing_counts > 0]
if not missing_counts.empty:
    sns.barplot(x=missing_counts.values, y=missing_counts.index, ax=axes[0, 0], color='#d97706')
    axes[0, 0].set_title('Missing values by column')
else:
    axes[0, 0].text(0.5, 0.5, 'No missing values', ha='center', va='center')
    axes[0, 0].set_axis_off()

if {'source', 'label'}.issubset(raw_df.columns):
    coverage = raw_df.assign(label_present=raw_df['label'].notna()).groupby('source', dropna=False)['label_present'].mean().sort_values(ascending=False).head(10)
    if not coverage.empty:
        sns.barplot(x=coverage.values, y=coverage.index.astype(str), ax=axes[0, 1], color='#2563eb')
        axes[0, 1].set_title('Label coverage by source')
        axes[0, 1].set_xlim(0, 1)
    else:
        axes[0, 1].text(0.5, 0.5, 'No source coverage data', ha='center', va='center')
        axes[0, 1].set_axis_off()
elif 'source' in raw_df.columns:
    source_counts = raw_df['source'].astype(str).value_counts().head(10)
    sns.barplot(x=source_counts.values, y=source_counts.index, ax=axes[0, 1], color='#2563eb')
    axes[0, 1].set_title('Top sources')
else:
    axes[0, 1].text(0.5, 0.5, 'No source column', ha='center', va='center')
    axes[0, 1].set_axis_off()

numeric_columns = raw_df.select_dtypes(include=['number']).columns.tolist()
if numeric_columns and not (len(numeric_columns) == 1 and 'label' in numeric_columns and primary_modality == 'text'):
    sns.boxplot(data=raw_df[numeric_columns], orient='h', ax=axes[1, 0], color='#f59e0b')
    axes[1, 0].set_title('Raw numeric distributions')
    sns.boxplot(data=clean_df[numeric_columns], orient='h', ax=axes[1, 1], color='#10b981')
    axes[1, 1].set_title('Cleaned numeric distributions')
else:
    if 'text' in raw_df.columns:
        raw_lengths = raw_df['text'].fillna('').astype(str).str.split().str.len()
        clean_lengths = clean_df['text'].fillna('').astype(str).str.split().str.len()
        sns.histplot(raw_lengths, bins=30, ax=axes[1, 0], color='#f59e0b')
        axes[1, 0].set_title('Raw text length distribution')
        sns.histplot(clean_lengths, bins=30, ax=axes[1, 1], color='#10b981')
        axes[1, 1].set_title('Cleaned text length distribution')
    else:
        axes[1, 0].text(0.5, 0.5, 'No numeric columns', ha='center', va='center')
        axes[1, 0].set_axis_off()
        axes[1, 1].text(0.5, 0.5, 'No numeric columns', ha='center', va='center')
        axes[1, 1].set_axis_off()

plt.tight_layout()
plt.show()
